In [22]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import train_test_split, LeaveOneGroupOut, HalvingGridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder, LabelBinarizer
import os
import sys
import datetime
sys.path.append(os.path.abspath(os.path.join('..', 'ml-app')))
from preprocessing import WindowingMethod, FeatureExtractor
from pipeline import CustomPipeline

In [2]:
random_state = 42
data_path = os.path.join('..', 'data')
training_path = os.path.join(data_path, 'training', 'converted')
labeled = [p for p in os.listdir(training_path) if p.endswith('.csv') and p.startswith('series_1')]
unlabeled = [p for p in os.listdir(training_path) if p.endswith('.csv') and p.startswith('series_2')]
model_path = os.path.join(data_path, 'models')
training_path, labeled, unlabeled, model_path

('../data/training/converted',
 ['series_1_sausage_4.csv',
  'series_1_sausage_2.csv',
  'series_1_sausage_3.csv',
  'series_1_sausage_5.csv',
  'series_1_sausage_1.csv'],
 ['series_2_sausage_5.csv',
  'series_2_sausage_2.csv',
  'series_2_sausage_1.csv',
  'series_2_sausage_3.csv',
  'series_2_sausage_4.csv'],
 '../data/models')

In [3]:
sample_data = pd.read_csv(os.path.join(training_path, labeled[0]))
feature_order = sample_data.columns.tolist()
feature_order.remove('label')
feature_order.remove('ws202_light_status')
feature_order.remove('ws202_pir_status')
feature_order

['timestamp',
 'sensor',
 'lse01_conductivity',
 'lse01_moisture',
 'lse01_temperature',
 'ltcfs_temperature']

In [4]:
classifiers = {
    "Logistic Regression": (
        "logreg",
        LogisticRegression(random_state=random_state),
        {
            "classifier__solver": ["liblinear", "lbfgs", "saga", "newton-cg"],
            "classifier__l1_ratio": [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1],
            "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "classifier__class_weight": [None, "balanced"],
            "classifier__max_iter": [1000, 2000],
        },
    ),
    "Random Forest": (
        "rf",
        RandomForestClassifier(random_state=random_state),
        {
            "classifier__n_estimators": [100, 200, 300, 500, 800],
            "classifier__max_depth": [None, 4, 6, 8, 12, 16, 24],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 4, 8],
            "classifier__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8, None],
            "classifier__bootstrap": [True],
            "classifier__class_weight": [None, "balanced", "balanced_subsample"],
            "classifier__criterion": ["gini", "entropy", "log_loss"],
        },
    ),
    "MLP Classifier": (
        "mlp",
        MLPClassifier(random_state=random_state),
        {
            "classifier__hidden_layer_sizes": [
                (8,),
                (16,),
                (32,),
                (64,),
                (32, 16),
                (64, 32),
                (64, 32, 16),
                (128, 64),
            ],
            "classifier__activation": ["relu", "tanh"],
            "classifier__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
            "classifier__learning_rate_init": [1e-4, 3e-4, 1e-3, 3e-3],
            "classifier__batch_size": [16, 32, 64, 128],
            "classifier__max_iter": [500, 1000, 2000],
            "classifier__early_stopping": [True],
            "classifier__validation_fraction": [0.1, 0.15, 0.2],
        },
    ),
}

In [5]:
def create_pipeline(classifier_name: str) -> CustomPipeline:
    if classifier_name not in classifiers:
        raise ValueError(
            f"Classifier '{classifier_name}' not found. Available classifiers: {list(classifiers.keys())}"
        )

    classifier_key, classifier, _ = classifiers[classifier_name]

    pipeline = CustomPipeline(
        [
            (
                "windowing",
                WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier", 
                classifier
            ),
        ],
        version=f"{classifier_key}_{datetime.datetime.now().strftime('%Y%m%d')}_v1",
        model_type=classifier_name,
        feature_order=feature_order,
    )

    return pipeline

In [6]:
pipelines = {
    name: create_pipeline(name) for name in classifiers.keys()
}

In [7]:
for name, pipeline in pipelines.items():
    print(f"Pipeline for {name}:")
    pipeline.display()
    print("\n" + "="*50 + "\n")

Pipeline for Logistic Regression:
Pipeline Version: logreg_20260520_v1
Model Type: Logistic Regression
Feature Order: ['timestamp', 'sensor', 'lse01_conductivity', 'lse01_moisture', 'lse01_temperature', 'ltcfs_temperature']
Created At: 2026-05-20T15:27:56.006091
Pipeline Steps:
 - windowing: WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2)
 - scaler: StandardScaler()
 - classifier: LogisticRegression(random_state=42)


Pipeline for Random Forest:
Pipeline Version: rf_20260520_v1
Model Type: Random Forest
Feature Order: ['timestamp', 'sensor', 'lse01_conductivity', 'lse01_moisture', 'lse01_temperature', 'ltcfs_temperature']
Created At: 2026-05-20T15:27:56.006091
Pipeline Steps:
 - windowing: WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2)
 - scaler: StandardScaler()
 - classifier: RandomForestClassifier(random_state=42)


Pipeline for MLP Classifier:
Pipeline Version: mlp_20260520_v1
Model Type: MLP Classifier
Feature Order: ['timestamp', 'sensor

In [19]:
labeled_data = []
for file in labeled:
    data = pd.read_csv(os.path.join(training_path, file))
    data['group'] = file.split('.')[0]
    labeled_data.append(data)
labeled_df = pd.concat(labeled_data, ignore_index=True)
labeled_df.sort_values(by='timestamp', inplace=True)
labeled_df.head(), labeled_df.shape

(                               timestamp sensor  lse01_conductivity  \
 194638  2023-03-30 00:01:27.651646+00:00  lse01              5930.0   
 259509  2023-03-30 00:01:39.515132+00:00  lse01              3851.0   
 0       2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 129764  2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 129765  2023-03-30 00:03:38.046627+00:00  ws202                 NaN   
 
         lse01_moisture  lse01_temperature  ltcfs_temperature  \
 194638            6.93               9.78                NaN   
 259509            1.46               9.88                NaN   
 0                  NaN                NaN                NaN   
 129764             NaN                NaN                NaN   
 129765             NaN                NaN                NaN   
 
         ws202_light_status  ws202_pir_status      label               group  
 194638                 NaN               NaN  Umroetung  series_1_sausage_5  
 259509         

In [17]:
group_count = labeled_df['group'].nunique()
group_cv = GroupKFold(n_splits=group_count)
splits = list(group_cv.split(labeled_df[feature_order], labeled_df['label'], groups=labeled_df['group']))
splits

[(array([     0,      1,      2, ..., 324374, 324375, 324376],
        shape=(259492,)),
  array([    13,     14,     15, ..., 324360, 324366, 324370],
        shape=(64885,))),
 (array([     0,      1,      3, ..., 324372, 324374, 324375],
        shape=(259498,)),
  array([     2,     26,     27, ..., 324367, 324373, 324376],
        shape=(64879,))),
 (array([     0,      1,      2, ..., 324373, 324374, 324376],
        shape=(259503,)),
  array([     3,      4,      5, ..., 324365, 324371, 324375],
        shape=(64874,))),
 (array([     1,      2,      3, ..., 324374, 324375, 324376],
        shape=(259506,)),
  array([     0,     20,     21, ..., 324362, 324369, 324372],
        shape=(64871,))),
 (array([     0,      2,      3, ..., 324373, 324375, 324376],
        shape=(259509,)),
  array([     1,      9,     10, ..., 324364, 324368, 324374],
        shape=(64868,)))]

In [ ]:
hyperparameter_selections = {}
hpo_scores = {}
for name, (key, classifier, param_grid) in classifiers.items():
    hyperparameter_selections[name] = {k: [] for k in param_grid.keys()}
    hpo_scores[name] = []

hyperparameter_selections

{'Logistic Regression': {'classifier__solver': [],
  'classifier__l1_ratio': [],
  'classifier__C': [],
  'classifier__class_weight': [],
  'classifier__max_iter': []},
 'Random Forest': {'classifier__n_estimators': [],
  'classifier__max_depth': [],
  'classifier__min_samples_split': [],
  'classifier__min_samples_leaf': [],
  'classifier__max_features': [],
  'classifier__bootstrap': [],
  'classifier__class_weight': [],
  'classifier__criterion': []},
 'MLP Classifier': {'classifier__hidden_layer_sizes': [],
  'classifier__activation': [],
  'classifier__alpha': [],
  'classifier__learning_rate_init': [],
  'classifier__batch_size': [],
  'classifier__max_iter': [],
  'classifier__early_stopping': [],
  'classifier__validation_fraction': []}}

In [ ]:
for i, (train_idx, test_idx) in enumerate(splits):
    print(f"Processing fold {i+1}/{len(splits)}...")
    X_train, y_train = labeled_df.iloc[train_idx][feature_order], labeled_df.iloc[train_idx]['label']
    X_test, y_test = labeled_df.iloc[test_idx][feature_order], labeled_df.iloc[test_idx]['label']
    
    for name, pipeline in pipelines.items():
        print(f"  Tuning hyperparameters for {name}...")
        serach = HalvingGridSearchCV(
            estimator=pipeline,
            param_grid=classifiers[name][2],
            factor=3,
            random_state=random_state,
            n_jobs=-1,
        )
        serach.fit(X_train, y_train)
        best_params = serach.best_params_
        for param, value in best_params.items():
            hyperparameter_selections[name][param].append(value)
        hpo_scores[name].append(serach.best_score_)
        print(f"    Best hyperparameters: {best_params}")
    
hyperparameter_selections, hpo_scores

In [ ]:
best_hyperparameters = {name: {} for name in classifiers.keys()}
for name, params in hyperparameter_selections.items():
    best_scores = hpo_scores[name]
    for param, values in params.items():
        vals = {}
        for val in values:
            if val in vals.keys():
                continue
            scores = [best_scores[i] for i in range(len(values)) if values[i] == val]
            vals[val] = np.sum(scores)
        best_val = max(vals, key=vals.get)
        best_hyperparameters[name][param] = best_va

best_hyperparameters